# Loan Default Prediction

## Phase 3: Model Training

### Objective

The objective of this notebook is to train and compare multiple classification
models to predict loan approval, using the cleaned dataset from Phase 2.

In this notebook, we will:
- Split the data into training and testing sets
- Train three models: Logistic Regression, Random Forest, and XGBoost
- Evaluate each model using Accuracy, Precision, Recall, F1-score, and ROC-AUC
- Compare results and select the best-performing model

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [3]:
df = pd.read_csv('../data/cleaned_train.csv')
df.shape

(614, 13)

In [4]:
df.columns.tolist()

['Gender',
 'Married',
 'Dependents',
 'Education',
 'Self_Employed',
 'ApplicantIncome',
 'CoapplicantIncome',
 'LoanAmount',
 'Loan_Amount_Term',
 'Credit_History',
 'Loan_Status',
 'Property_Area_Semiurban',
 'Property_Area_Urban']

### Train-Test Split

The data is split into 80% training and 20% testing sets. Stratified sampling
is used to preserve the same approve/reject ratio in both sets, since the target
variable is imbalanced.

In [5]:
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (491, 12)
Testing set: (123, 12)


### Train Models

Three models are trained on the same training data for a fair comparison:
Logistic Regression, Random Forest, and XGBoost.

In [6]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42)
}

trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"{name} trained successfully")

C:\Users\srika\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression trained successfully
Random Forest trained successfully
XGBoost trained successfully


### Evaluate Models

Each model is evaluated on the test set using Accuracy, Precision, Recall,
F1-score, and ROC-AUC. Accuracy alone is not enough here since the target
variable is imbalanced (~69% approved vs ~31% rejected).

In [7]:
results = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    }

results_df = pd.DataFrame(results).T
results_df.round(3)

,Accuracy,Precision,Recall,F1,ROC-AUC
Logistic Regression,0.862,0.840,0.988,0.908,0.849
Random Forest,0.821,0.839,0.918,0.876,0.809
XGBoost,0.789,0.831,0.871,0.851,0.780


### Save Best Model

Logistic Regression performed best overall (highest Accuracy, F1, and ROC-AUC),
likely because the dataset is small and the relationships between features and
approval are fairly linear. It is saved for use in the explainability phase.

In [8]:
import joblib

best_model = trained_models["Logistic Regression"]
joblib.dump(best_model, '../models/logistic_regression_model.pkl')
print("Model saved successfully")

Model saved successfully


### Summary

- Three models were trained: Logistic Regression, Random Forest, and XGBoost.
- Logistic Regression achieved the best overall performance across all metrics.
- The trained model is saved as `logistic_regression_model.pkl` for use in
  Phase 4 (SHAP Explainability).